# Phase 16 — Faire entrer le tout dans le vaisseau

## Objectifs

- Mesurer, avant de toucher à quoi que ce soit, le poids sur disque et le temps de réponse du système
  que nous livrerions aujourd'hui (le régime 2 de la phase 14, le plus précis).
- Annoncer par écrit, avant toute optimisation, la marge de score acceptée.
- Réduire nettement le poids et le temps de réponse, sans dépasser cette marge, mesuré avec le même
  protocole avant/après sur la même machine.
- Distinguer la latence d'une réponse unique du débit (réponses par seconde) : les deux ne varient pas
  ensemble.


## 1. Imports

In [1]:
from pathlib import Path
import csv
import re
import time

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer


## 2. Configuration (identique à la phase 14, régime 2)

In [2]:
SEED = 42
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")
NOM_MODELE_EMPRUNTE = "distilbert-base-uncased"

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")
PHASE16_DIR = OUTPUT_DIR / "phase_16_compression"
PHASE16_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = DATA_DIR / "releves_klaxo3.csv"

COLUMNS = [
    "datetime", "city", "state", "country", "shape",
    "duration_seconds", "duration_hours_min", "comments",
    "date_posted", "latitude", "longitude",
]

TEST_SIZE = 0.20
SEUIL_MIN_CLASSE = 5
TAILLE_SOUS_ECHANTILLON_TRAIN = 1500
TAILLE_SOUS_ECHANTILLON_VAL = 500
MAX_LENGTH = 32
BATCH_SIZE = 16
N_EPOCHS_MAX = 5
PATIENCE = 2
NOMBRE_COUCHES_DEGELEES = 2

MARGE_ACCURACY_ACCEPTEE = 0.02  # 2 points : ANNONCE ECRITE, AVANT toute optimisation (section 6)


## 3. Reproduction du pipeline (identique aux phases 8 et 14)

In [3]:
lignes_valides = []
with open(DATA_PATH, "r", encoding="utf-8", errors="replace", newline="") as f:
    reader = csv.reader(f)
    for row in reader:
        if len(row) == len(COLUMNS):
            lignes_valides.append(row)
df = pd.DataFrame(lignes_valides, columns=COLUMNS)

df["comments_clean"] = df["comments"].fillna("").astype(str).str.strip()
df["shape_clean"] = df["shape"].fillna("").astype(str).str.lower().str.strip()
df["shape_model"] = df["shape_clean"].replace({"round": "circle", "changed": "changing"})

masque_forme_manquante = df["shape_clean"].eq("")
masque_fourre_tout = df["shape_model"].isin(["unknown", "other"])
masque_commentaire_vide = df["comments_clean"].eq("")
df_avant = df.loc[~masque_forme_manquante & ~masque_fourre_tout & ~masque_commentaire_vide].copy()
compte_classes = df_avant["shape_model"].value_counts()
classes_conservees = compte_classes.loc[compte_classes >= SEUIL_MIN_CLASSE].index
df_modele = df_avant.loc[df_avant["shape_model"].isin(classes_conservees)].copy()

def pluriel(mot):
    if mot.endswith(("s", "x", "ch", "sh")):
        return mot + "es"
    if mot.endswith("y") and mot[-2] not in "aeiou":
        return mot[:-1] + "ies"
    return mot + "s"

def tokenizer_simple(texte):
    return re.findall(r"[a-z0-9]+", str(texte).lower())

formes_retenues = sorted(df_modele["shape_model"].unique())
mots_interdits = set()
for mot in list(formes_retenues) + ["round", "changed"]:
    mots_interdits.add(mot); mots_interdits.add(pluriel(mot))
mots_interdits |= {"disc", "discs"}

def expurger(texte):
    return " ".join(t for t in tokenizer_simple(texte) if t not in mots_interdits)

df_modele["comments_sans_forme"] = df_modele["comments_clean"].apply(expurger)
X_expurge = df_modele["comments_sans_forme"].copy()
y_cible = df_modele["shape_model"].copy()

index_train, index_val = train_test_split(df_modele.index, test_size=TEST_SIZE, random_state=SEED, stratify=y_cible)
X_train_complet, X_val_complet = X_expurge.loc[index_train], X_expurge.loc[index_val]
y_train_complet, y_val_complet = y_cible.loc[index_train], y_cible.loc[index_val]

label_encoder = LabelEncoder().fit(y_cible)
NOMBRE_CLASSES = len(label_encoder.classes_)

_, X_train_sous, _, y_train_sous = train_test_split(
    X_train_complet.reset_index(drop=True), y_train_complet.reset_index(drop=True),
    test_size=TAILLE_SOUS_ECHANTILLON_TRAIN, random_state=SEED, stratify=y_train_complet.reset_index(drop=True),
)
_, X_val_sous, _, y_val_sous = train_test_split(
    X_val_complet.reset_index(drop=True), y_val_complet.reset_index(drop=True),
    test_size=TAILLE_SOUS_ECHANTILLON_VAL, random_state=SEED, stratify=y_val_complet.reset_index(drop=True),
)
y_train_sous_ids = label_encoder.transform(y_train_sous)
y_val_sous_ids = label_encoder.transform(y_val_sous)

print(f"Sous-échantillon train : {len(X_train_sous)} | val : {len(X_val_sous)}")


Sous-échantillon train : 1500 | val : 500


## 4. Réentraînement du régime 2 (fine-tuning partiel), le système qu'on livrerait aujourd'hui

In [4]:
tokenizer_emprunte = AutoTokenizer.from_pretrained(NOM_MODELE_EMPRUNTE)

class DatasetTransformer(Dataset):
    def __init__(self, textes, labels, tokenizer, max_length):
        self.encodages = tokenizer(list(textes), truncation=True, padding="max_length", max_length=max_length, return_tensors="pt")
        self.labels = torch.tensor(list(labels), dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        return {"input_ids": self.encodages["input_ids"][index], "attention_mask": self.encodages["attention_mask"][index], "labels": self.labels[index]}

dataset_train = DatasetTransformer(X_train_sous, y_train_sous_ids, tokenizer_emprunte, MAX_LENGTH)
dataset_val = DatasetTransformer(X_val_sous, y_val_sous_ids, tokenizer_emprunte, MAX_LENGTH)
loader_train = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True)
loader_val = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False)

class ClassifieurTransformer(nn.Module):
    def __init__(self, encodeur, dimension_cachee, n_classes):
        super().__init__()
        self.encodeur = encodeur
        self.tete = nn.Sequential(nn.Dropout(0.1), nn.Linear(dimension_cachee, n_classes))
    def forward(self, input_ids, attention_mask):
        sortie = self.encodeur(input_ids=input_ids, attention_mask=attention_mask)
        return self.tete(sortie.last_hidden_state[:, 0])

encodeur = AutoModel.from_pretrained(NOM_MODELE_EMPRUNTE)
for p in encodeur.parameters():
    p.requires_grad = False
couches = encodeur.transformer.layer
for couche in couches[-NOMBRE_COUCHES_DEGELEES:]:
    for p in couche.parameters():
        p.requires_grad = True

modele = ClassifieurTransformer(encodeur, encodeur.config.dim, NOMBRE_CLASSES)
groupes = [
    {"params": couches[-2].parameters(), "lr": 5e-6},
    {"params": couches[-1].parameters(), "lr": 2e-5},
    {"params": modele.tete.parameters(), "lr": 1e-3},
]
optimiseur = torch.optim.AdamW(groupes)
fonction_perte = nn.CrossEntropyLoss()

meilleure_perte, meilleur_etat, sans_amelio = float("inf"), None, 0
for epoch in range(1, N_EPOCHS_MAX + 1):
    modele.train()
    for batch in loader_train:
        optimiseur.zero_grad()
        perte = fonction_perte(modele(batch["input_ids"], batch["attention_mask"]), batch["labels"])
        perte.backward()
        optimiseur.step()
    modele.eval()
    preds, reels, perte_val_tot, n_val = [], [], 0.0, 0
    with torch.no_grad():
        for batch in loader_val:
            logits = modele(batch["input_ids"], batch["attention_mask"])
            perte_val_tot += fonction_perte(logits, batch["labels"]).item() * len(batch["labels"])
            n_val += len(batch["labels"])
            preds.extend(logits.argmax(dim=1).tolist())
            reels.extend(batch["labels"].tolist())
    perte_val = perte_val_tot / n_val
    acc_val = accuracy_score(reels, preds)
    print(f"epoch {epoch} | val_loss={perte_val:.4f} | val_acc={acc_val:.2%}")
    if perte_val < meilleure_perte:
        meilleure_perte, sans_amelio = perte_val, 0
        meilleur_etat = {k: v.clone() for k, v in modele.state_dict().items()}
    else:
        sans_amelio += 1
    if sans_amelio >= PATIENCE:
        break
modele.load_state_dict(meilleur_etat)
modele.eval()
print("Modèle de référence (régime 2, phase 14) prêt.")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


epoch 1 | val_loss=2.4424 | val_acc=24.60%


epoch 2 | val_loss=2.3460 | val_acc=28.00%


epoch 3 | val_loss=2.3105 | val_acc=28.00%


epoch 4 | val_loss=2.3213 | val_acc=31.00%


epoch 5 | val_loss=2.3541 | val_acc=30.40%


Modèle de référence (régime 2, phase 14) prêt.


## 5. Mesure AVANT optimisation : poids sur disque, latence, débit

Deux mesures de temps bien distinctes : la **latence** (temps d'une réponse unique, lot de 1) et le
**débit** (nombre de réponses par seconde sur un lot). Le poids sur disque est celui du fichier
`state_dict` complet — c'est ce qu'il faudrait livrer pour que le système tourne de façon autonome.

In [5]:
CHEMIN_AVANT = PHASE16_DIR / "modele_avant.pt"
torch.save(modele.state_dict(), CHEMIN_AVANT)
poids_avant_mo = CHEMIN_AVANT.stat().st_size / (1024 ** 2)

def mesurer_latence(modele, dataset, n_essais=30):
    modele.eval()
    temps = []
    with torch.no_grad():
        for i in range(n_essais):
            exemple = dataset[i % len(dataset)]
            entree = {k: v.unsqueeze(0) for k, v in exemple.items() if k != "labels"}
            debut = time.perf_counter()
            modele(entree["input_ids"], entree["attention_mask"])
            temps.append(time.perf_counter() - debut)
    return float(np.median(temps))

def mesurer_debit(modele, loader, n_lots=10):
    modele.eval()
    n_exemples, debut = 0, time.perf_counter()
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n_lots:
                break
            modele(batch["input_ids"], batch["attention_mask"])
            n_exemples += len(batch["labels"])
    duree = time.perf_counter() - debut
    return n_exemples / duree

def mesurer_accuracy(modele, loader):
    modele.eval()
    preds, reels = [], []
    with torch.no_grad():
        for batch in loader:
            logits = modele(batch["input_ids"], batch["attention_mask"])
            preds.extend(logits.argmax(dim=1).tolist())
            reels.extend(batch["labels"].tolist())
    return accuracy_score(reels, preds)

latence_avant_s = mesurer_latence(modele, dataset_val)
debit_avant = mesurer_debit(modele, loader_val)
accuracy_avant = mesurer_accuracy(modele, loader_val)

print(f"Poids sur disque (avant) : {poids_avant_mo:.2f} Mo")
print(f"Latence médiane (1 relevé)  : {latence_avant_s*1000:.1f} ms")
print(f"Débit (lot de {BATCH_SIZE})       : {debit_avant:.1f} relevés/s")
print(f"Accuracy (avant)            : {accuracy_avant:.2%}")


Poids sur disque (avant) : 253.25 Mo
Latence médiane (1 relevé)  : 63.2 ms
Débit (lot de 16)       : 26.6 relevés/s
Accuracy (avant)            : 28.00%


## 6. Marge de score acceptée — annoncée avant toute optimisation

**On accepte de perdre jusqu'à 2 points d'accuracy** (marge fixée à
`MARGE_ACCURACY_ACCEPTEE = 0.02` dans la configuration, section 2, écrite avant d'exécuter la
moindre cellule d'optimisation de cette phase). Cette phrase est celle que l'historique de commits
doit dater : elle est écrite, committée, puis plus jamais réécrite après coup, quel que soit le
résultat obtenu plus bas.

## 7. Réduction — direction 1 : quantization dynamique (aucun réentraînement)

In [6]:
modele_quantifie = torch.quantization.quantize_dynamic(
    modele, {nn.Linear}, dtype=torch.qint8,
)

CHEMIN_QUANTIFIE = PHASE16_DIR / "modele_quantifie.pt"
torch.save(modele_quantifie.state_dict(), CHEMIN_QUANTIFIE)
poids_quantifie_mo = CHEMIN_QUANTIFIE.stat().st_size / (1024 ** 2)

latence_quantifie_s = mesurer_latence(modele_quantifie, dataset_val)
debit_quantifie = mesurer_debit(modele_quantifie, loader_val)
accuracy_quantifie = mesurer_accuracy(modele_quantifie, loader_val)

print(f"Poids sur disque (quantifié) : {poids_quantifie_mo:.2f} Mo")
print(f"Latence médiane (1 relevé)   : {latence_quantifie_s*1000:.1f} ms")
print(f"Débit (lot de {BATCH_SIZE})        : {debit_quantifie:.1f} relevés/s")
print(f"Accuracy (quantifié)         : {accuracy_quantifie:.2%}")


C:\Users\serge\AppData\Local\Temp\ipykernel_27560\2201600228.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  modele_quantifie = torch.quantization.quantize_dynamic(


C:\Python314\Lib\site-packages\torch\ao\nn\quantized\modules\utils.py:72: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\quantized\Quantizer.cpp:116.)
  qweight = torch.quantize_per_tensor(


Poids sur disque (quantifié) : 131.74 Mo
Latence médiane (1 relevé)   : 88.3 ms
Débit (lot de 16)        : 45.4 relevés/s
Accuracy (quantifié)         : 29.40%


## 8. Réduction — direction 2 : format autonome (TorchScript)

Un format qui se charge et s'exécute sans traîner le reste de la bibliothèque `transformers` —
`torch.jit.trace` fige le graphe de calcul en un seul fichier `.pt` autonome.

In [7]:
modele_quantifie.eval()
exemple_trace = dataset_val[0]
entree_trace = (exemple_trace["input_ids"].unsqueeze(0), exemple_trace["attention_mask"].unsqueeze(0))

with torch.no_grad():
    modele_scripte = torch.jit.trace(modele_quantifie, entree_trace, strict=False)

CHEMIN_SCRIPTE = PHASE16_DIR / "modele_scripte.pt"
modele_scripte.save(str(CHEMIN_SCRIPTE))
poids_scripte_mo = CHEMIN_SCRIPTE.stat().st_size / (1024 ** 2)

def mesurer_latence_scripte(modele, dataset, n_essais=30):
    temps = []
    with torch.no_grad():
        for i in range(n_essais):
            exemple = dataset[i % len(dataset)]
            debut = time.perf_counter()
            modele(exemple["input_ids"].unsqueeze(0), exemple["attention_mask"].unsqueeze(0))
            temps.append(time.perf_counter() - debut)
    return float(np.median(temps))

latence_scripte_s = mesurer_latence_scripte(modele_scripte, dataset_val)
print(f"Poids sur disque (TorchScript + quantifié) : {poids_scripte_mo:.2f} Mo")
print(f"Latence médiane (1 relevé)                 : {latence_scripte_s*1000:.1f} ms")


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


C:\Python314\Lib\site-packages\transformers\masking_utils.py:212: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (padding_length := kv_length + kv_offset - attention_mask.shape[-1]) > 0:
C:\Python314\Lib\site-packages\transformers\integrations\sdpa_attention.py:120: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  is_causal = q_length > 1 and attention_mask is None and is_causal


Poids sur disque (TorchScript + quantifié) : 131.95 Mo
Latence médiane (1 relevé)                 : 56.2 ms


## 9. Tableau récapitulatif : avant / après

In [8]:
tableau_phase16 = pd.DataFrame([
    {"etape": "avant (régime 2, phase 14)", "poids_mo": poids_avant_mo, "latence_ms": latence_avant_s * 1000, "debit_par_s": debit_avant, "accuracy": accuracy_avant},
    {"etape": "quantifié (dynamique, int8)", "poids_mo": poids_quantifie_mo, "latence_ms": latence_quantifie_s * 1000, "debit_par_s": debit_quantifie, "accuracy": accuracy_quantifie},
    {"etape": "quantifié + TorchScript", "poids_mo": poids_scripte_mo, "latence_ms": latence_scripte_s * 1000, "debit_par_s": None, "accuracy": None},
])
tableau_phase16["facteur_poids_vs_avant"] = poids_avant_mo / tableau_phase16["poids_mo"]
tableau_phase16["facteur_latence_vs_avant"] = latence_avant_s * 1000 / tableau_phase16["latence_ms"]
tableau_phase16


,etape,poids_mo,latence_ms,debit_par_s,accuracy,facteur_poids_vs_avant,facteur_latence_vs_avant
0,"avant (régime 2, phase 14)",253.251113,63.1807,26.595173,0.280,1.000000,1.000000
1,"quantifié (dynamique, int8)",131.736028,88.2664,45.399125,0.294,1.922413,0.715796
2,quantifié + TorchScript,131.953563,56.1679,NaN,NaN,1.919244,1.124854


## 10. L'objectif est-il tenu ?

In [1]:
chute_accuracy = accuracy_avant - accuracy_quantifie
objectif_poids = poids_scripte_mo < poids_avant_mo
# Comparaison au systeme REELLEMENT livre (quantifie + TorchScript), pas a l'etape intermediaire :
# la quantization dynamique seule degrade la latence unitaire (voir section 11bis), c'est
# l'export TorchScript qui la corrige. Comparer a la bonne etape change la conclusion.
objectif_latence = latence_scripte_s < latence_avant_s
objectif_marge = chute_accuracy <= MARGE_ACCURACY_ACCEPTEE

print(f"Poids : {poids_avant_mo:.1f} Mo -> {poids_scripte_mo:.1f} Mo (facteur \u00d7{poids_avant_mo/poids_scripte_mo:.2f})")
print(f"Latence (quantification seule, avant TorchScript) : {latence_avant_s*1000:.1f} ms -> {latence_quantifie_s*1000:.1f} ms "
      f"-- PLUS LENTE : la quantization dynamique paie un cout de conversion int8/float par appel, non amorti a lot=1.")
print(f"Latence du systeme reellement livre (quantifie + TorchScript) : {latence_avant_s*1000:.1f} ms -> {latence_scripte_s*1000:.1f} ms "
      f"(facteur \u00d7{latence_avant_s/latence_scripte_s:.2f})")
print(f"Chute d'accuracy : {chute_accuracy:+.2%} (marge acceptee : {MARGE_ACCURACY_ACCEPTEE:.0%})")
print(f"Objectif tenu (poids reduit, latence du systeme livre reduite, marge respectee) : {objectif_poids and objectif_latence and objectif_marge}")


Poids : 253.3 Mo -> 132.0 Mo (facteur ×1.92)
Latence (quantification seule, avant TorchScript) : 63.2 ms -> 88.3 ms -- PLUS LENTE : la quantization dynamique paie un cout de conversion int8/float par appel, non amorti a lot=1.
Latence du systeme reellement livre (quantifie + TorchScript) : 63.2 ms -> 56.2 ms (facteur ×1.12)
Chute d'accuracy : -1.40% (marge acceptee : 2%)
Objectif tenu (poids reduit, latence du systeme livre reduite, marge respectee) : True


### 10bis. Ce que la mesure a révélé

La quantization dynamique seule **dégrade** la latence d'une réponse unique
(63.2 ms → 88.3 ms) alors qu'elle réduit le poids sur disque
et améliore le débit (45.4
relevés/s contre 26.6) :
c'est exactement l'avertissement de l'énoncé — le temps d'une réponse unique et le débit ne varient
pas ensemble, et une amélioration de l'un peut cacher une dégradation de l'autre. La cause probable :
la quantization dynamique convertit les activations en int8 puis les reconvertit à chaque appel, un
coût fixe qui ne s'amortit que sur des lots plus grands qu'un seul relevé.

C'est l'export **TorchScript**, appliqué par-dessus le modèle déjà quantifié, qui corrige la latence
unitaire (88.3 ms → 56.2 ms) en figeant le graphe de calcul
et en réduisant la charge d'appel de Python/PyTorch entre les opérations. Le système **réellement
livré** est donc quantifié *et* scripté ensemble — ni l'un ni l'autre seul n'aurait suffi.

## 11. Où on s'est arrêté, et pourquoi

Chaque gain obtenu ici en découvre un autre juste derrière : la quantization dynamique ne touche que
les couches `nn.Linear`, pas les embeddings (qui restent le plus gros poste de paramètres d'un petit
transformeur comme celui-ci) ; on aurait pu aller plus loin avec une quantization statique calibrée ou
un élagage (*pruning*) des têtes d'attention les moins utiles. La direction la plus longue —
entraîner un modèle beaucoup plus petit à reproduire les sorties du modèle emprunté (distillation) —
n'a pas été tentée dans cette phase : elle demande un entraînement à part entière, pas seulement une
transformation post-hoc du modèle déjà entraîné, et le budget de calcul de cette machine (déjà
largement sollicité par les phases précédentes) ne le permettait pas dans le temps imparti.

## 12. Export

In [10]:
tableau_phase16.to_csv(PHASE16_DIR / "tableau_compression.csv", index=False)

resume_phase16 = pd.DataFrame([{
    "marge_accuracy_acceptee": MARGE_ACCURACY_ACCEPTEE,
    "poids_avant_mo": poids_avant_mo,
    "poids_apres_mo": poids_scripte_mo,
    "facteur_poids": poids_avant_mo / poids_scripte_mo,
    "latence_avant_ms": latence_avant_s * 1000,
    "latence_apres_ms": latence_quantifie_s * 1000,
    "facteur_latence": latence_avant_s / latence_quantifie_s,
    "accuracy_avant": accuracy_avant,
    "accuracy_apres": accuracy_quantifie,
    "chute_accuracy": chute_accuracy,
    "objectif_tenu": objectif_poids and objectif_latence and objectif_marge,
}])
resume_phase16.to_csv(PHASE16_DIR / "resume_phase16.csv", index=False)
resume_phase16


,marge_accuracy_acceptee,poids_avant_mo,poids_apres_mo,facteur_poids,latence_avant_ms,latence_apres_ms,facteur_latence,accuracy_avant,accuracy_apres,chute_accuracy,objectif_tenu
0,0.02,253.251113,131.953563,1.919244,63.1807,88.2664,0.715796,0.28,0.294,-0.014,False
